<div style="background: linear-gradient(90deg, #020B18 0%, #061A33 50%, #008CFF 100%); padding: 24px; border-radius: 12px; text-align: center;">
  <h1 style="color: #F4F8FC; margin: 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, 'Helvetica Neue', Arial; font-weight: 700; font-size: 2.2rem;">
    Students Performance in Exams
  </h1>
</div>

 # Students Performance in Exams — Exploratory Data Analysis

> **Objective:** Explore the factors associated with students' academic performance and test whether completing a test-preparation course is associated with higher total scores.

This notebook covers **data inspection, data quality checks, feature engineering, exploratory analysis, visualization, outlier detection, and hypothesis testing**.

### Dataset
The analysis uses the **Students Performance in Exams** dataset containing 1,000 student records and 8 original variables.

### Key questions
1. What does the overall score distribution look like?
2. How does performance vary by gender and race/ethnicity group?
3. Is parental education associated with student performance?
4. Do students who completed the test-preparation course score higher?
5. Is the difference in total scores statistically significant?

<div style="background: linear-gradient(90deg, #020B18 0%, #061A33 55%, #008CFF 100%); padding: 18px 22px; border-radius: 10px; text-align: left; border-left: 5px solid #00D9FF;">

  <h2 style="color: #F4F8FC; margin: 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial; font-weight: 700;">
    ANALYSIS ROADMAP
  </h2>

</div>


1. **Setup & Data Loading**
2. **Data Understanding & Quality Checks**
3. **Feature Engineering**
4. **Exploratory Data Analysis**
   - Test-preparation participation
   - Score distributions
   - Parental education
   - Gender
   - Race/ethnicity
5. **Outlier Analysis**
6. **Hypothesis Testing**
7. **Key Findings & Conclusion**


<div style="background: linear-gradient(90deg, #020B18 0%, #061A33 55%, #008CFF 100%); padding: 18px 22px; border-radius: 10px; text-align: left; border-left: 5px solid #00D9FF;">

  <h2 style="color: #F4F8FC; margin: 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial; font-weight: 700;">
    🔧 1. SETUP & DATA LOADING
  </h2>

</div>



In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import ttest_ind

import plotly.express as px

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

In [2]:
data = pd.read_csv(
    "/kaggle/input/datasets/spscientist/students-performance-in-exams/StudentsPerformance.csv"
)

data.head()

,gender,race/ethnicity,parental level of education,lunch,test preparation course,math score,reading score,writing score
0,female,group B,bachelor's degree,standard,none,72,72,74
1,female,group C,some college,standard,completed,69,90,88
2,female,group B,master's degree,standard,none,90,95,93
3,male,group A,associate's degree,free/reduced,none,47,57,44
4,male,group C,some college,standard,none,76,78,75





  
<div style="background: linear-gradient(90deg, #020B18 0%, #061A33 55%, #008CFF 100%); padding: 18px 22px; border-radius: 10px; text-align: left; border-left: 5px solid #00D9FF;">

  <h2 style="color: #F4F8FC; margin: 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial; font-weight: 700;">
    📊 2. DATA UNDERSTANDING & QUALITY CHECKS
  </h2>

</div>




In [3]:
data.shape

(1000, 8)

In [4]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column                       Non-Null Count  Dtype 
---  ------                       --------------  ----- 
 0   gender                       1000 non-null   object
 1   race/ethnicity               1000 non-null   object
 2   parental level of education  1000 non-null   object
 3   lunch                        1000 non-null   object
 4   test preparation course      1000 non-null   object
 5   math score                   1000 non-null   int64 
 6   reading score                1000 non-null   int64 
 7   writing score                1000 non-null   int64 
dtypes: int64(3), object(5)
memory usage: 62.6+ KB


In [5]:
data.describe()

,math score,reading score,writing score
count,"1,000.00","1,000.00","1,000.00"
mean,66.09,69.17,68.05
std,15.16,14.60,15.20
min,0.00,17.00,10.00
25%,57.00,59.00,57.75
50%,66.00,70.00,69.00
75%,77.00,79.00,79.00
max,100.00,100.00,100.00


In [6]:
missing_values = data.isna().sum().sort_values(ascending=False)
missing_values

gender                         0
race/ethnicity                 0
parental level of education    0
lunch                          0
test preparation course        0
math score                     0
reading score                  0
writing score                  0
dtype: int64

In [7]:
duplicate_count = data.duplicated().sum()
print(f"Duplicate rows: {duplicate_count}")

Duplicate rows: 0


In [8]:
data.nunique().sort_values()

gender                          2
lunch                           2
test preparation course         2
race/ethnicity                  5
parental level of education     6
reading score                  72
writing score                  77
math score                     81
dtype: int64



<div style="background: linear-gradient(135deg, #020B18 0%, #061A33 100%); padding: 20px; border-radius: 12px; border: 1px solid #008CFF;">

  <h3 style="color: #00D9FF; margin-top: 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial;">
    INITIAL OBSERVATIONS
  </h3>


  <p style="color: #F4F8FC; margin: 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial; line-height: 1.6;">
    - The dataset contains <strong>1,000 students</strong> and <strong>8 original variables</strong>.<br>
    - The dataset has <strong>no missing values</strong>.<br>
    - The categorical variables contain a manageable number of groups, making them suitable for grouped analysis.<br>
    - The three score variables are numeric and range from 0 to 100.
  </p>

</div>




  
<div style="background: linear-gradient(90deg, #020B18 0%, #061A33 55%, #008CFF 100%); padding: 18px 22px; border-radius: 10px; text-align: left; border-left: 5px solid #00D9FF;">

  <h2 style="color: #F4F8FC; margin: 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial; font-weight: 700;">
    ⚙️ 3. FEATURE ENGINEERING
  </h2>

</div>







In [9]:
score_columns = ["math score", "reading score", "writing score"]

data["average score"] = data[score_columns].mean(axis=1)
data["total score"] = data[score_columns].sum(axis=1)

data[score_columns + ["average score", "total score"]].head()

,math score,reading score,writing score,average score,total score
0,72,72,74,72.67,218
1,69,90,88,82.33,247
2,90,95,93,92.67,278
3,47,57,44,49.33,148
4,76,78,75,76.33,229


In [10]:
top_students = (
    data.sort_values("total score", ascending=False)
        .loc[:, [
            "gender",
            "race/ethnicity",
            "parental level of education",
            "test preparation course",
            *score_columns,
            "average score",
            "total score"
        ]]
)

top_students.head(10)

,gender,race/ethnicity,parental level of education,test preparation course,math score,reading score,writing score,average score,total score
916,male,group E,bachelor's degree,completed,100,100,100,100.00,300
962,female,group E,associate's degree,none,100,100,100,100.00,300
458,female,group E,bachelor's degree,none,100,100,100,100.00,300
114,female,group E,bachelor's degree,completed,99,100,100,99.67,299
712,female,group D,some college,none,98,100,99,99.00,297
179,female,group D,some high school,completed,97,100,100,99.00,297
165,female,group C,bachelor's degree,completed,96,100,100,98.67,296
625,male,group D,some college,completed,100,97,99,98.67,296
685,female,group E,master's degree,completed,94,99,100,97.67,293
903,female,group D,bachelor's degree,completed,93,100,100,97.67,293


In [11]:
top_students = (
    data.sort_values("total score", ascending=False)
        .loc[:, [
            "gender",
            "race/ethnicity",
            "parental level of education",
            "test preparation course",
            *score_columns,
            "average score",
            "total score"
        ]]
)

top_students.tail(10)

,gender,race/ethnicity,parental level of education,test preparation course,math score,reading score,writing score,average score,total score
211,male,group C,some college,none,35,28,27,30.00,90
787,female,group B,some college,none,19,38,32,29.67,89
338,female,group B,some high school,none,24,38,27,29.67,89
601,female,group C,high school,none,29,29,30,29.33,88
76,male,group E,some high school,none,30,26,22,26.00,78
17,female,group B,some high school,none,18,32,28,26.00,78
327,male,group A,some college,none,28,23,19,23.33,70
596,male,group B,high school,none,30,24,15,23.00,69
980,female,group B,high school,none,8,24,23,18.33,55
59,female,group C,some high school,none,0,17,10,9.00,27


<div style="background: linear-gradient(90deg, #020B18 0%, #061A33 55%, #008CFF 100%); padding: 18px 22px; border-radius: 10px; text-align: left; border-left: 5px solid #00D9FF;">

  <h2 style="color: #F4F8FC; margin: 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial; font-weight: 700;">
    📈 4. EXPLORATORY DATA ANALYSIS
  </h2>

</div>



<div style="background: linear-gradient(90deg, #061A33 0%, #020B18 100%); padding: 14px 20px; border-radius: 8px; text-align: left; border-left: 4px solid #008CFF;">

  <h3 style="color: #00D9FF; margin: 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial; font-weight: 700;">
     4.1 TEST-PREPARATION COURSE PARTICIPATION
  </h3>

</div>



In [12]:
prep_counts = data["test preparation course"].value_counts()

prep_counts

test preparation course
none         642
completed    358
Name: count, dtype: int64

In [13]:
fig = px.bar(
    prep_counts.reset_index(),
    x="test preparation course",
    y="count",
    title="Test-Preparation Course Participation",
    labels={
        "test preparation course": "Course Status",
        "count": "Number of Students"
    },
    text="count",
    color_discrete_sequence=["#008CFF"]
)

fig.update_traces(
    textposition="outside",
    marker=dict(
        line=dict(width=1, color="#00D9FF")
    )
)

fig.update_layout(
    showlegend=False,
    plot_bgcolor="#020B18",
    paper_bgcolor="#020B18",
    font=dict(color="#F4F8FC", family="Arial, sans-serif"),
    title=dict(
        text="Test-Preparation Course Participation",
        font=dict(color="#00D9FF", size=20)
    ),
    xaxis=dict(
        title=dict(text="Course Status", font=dict(color="#00D9FF")),
        tickfont=dict(color="#F4F8FC"),
        gridcolor="#061A33"
    ),
    yaxis=dict(
        title=dict(text="Number of Students", font=dict(color="#00D9FF")),
        tickfont=dict(color="#F4F8FC"),
        gridcolor="#061A33"
    ),
    margin=dict(l=40, r=40, t=60, b=40)
)

fig.show()

<div style="background: linear-gradient(135deg, #020B18 0%, #061A33 100%); padding: 20px; border-radius: 12px; border: 1px solid #008CFF;">

  <h3 style="color: #00D9FF; margin-top: 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial;">
    OBSERVATION
  </h3>

  <p style="color: #F4F8FC; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial; line-height: 1.6;">
    <strong> 358 students completed the preparation course, while 642 did not. Therefore, most students in this dataset did not complete the course.
  </p>

  <p style="color: #F4F8FC; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial;">
    <strong style="color: #00D9FF;">Completed:</strong> 358 students
  </p>

  <p style="color: #F4F8FC; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial;">
    <strong style="color: #00D9FF;">Not Completed:</strong> 642 students
  </p>

</div>

<div style="background: linear-gradient(90deg, #061A33 0%, #020B18 100%); padding: 14px 20px; border-radius: 8px; text-align: left; border-left: 4px solid #008CFF;">

  <h3 style="color: #00D9FF; margin: 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial; font-weight: 700;">
     4.2 OVERALL SCORE DISTRIBUTION
  </h3>

</div>

In [14]:
fig = px.histogram(
    data,
    x="total score",
    nbins=20,
    marginal="box",
    title="Distribution of Total Scores",
    labels={"total score": "Total Score"},
    color_discrete_sequence=["#008CFF"]
)

fig.update_traces(
    marker=dict(
        line=dict(width=1, color="#00D9FF")
    )
)

fig.update_layout(
    plot_bgcolor="#020B18",
    paper_bgcolor="#020B18",
    font=dict(color="#F4F8FC", family="Arial, sans-serif"),
    title=dict(
        text="Distribution of Total Scores",
        font=dict(color="#00D9FF", size=20)
    ),
    xaxis=dict(
        title=dict(text="Total Score", font=dict(color="#00D9FF")),
        tickfont=dict(color="#F4F8FC"),
        gridcolor="#061A33"
    ),
    yaxis=dict(
        title=dict(text="Count", font=dict(color="#00D9FF")),
        tickfont=dict(color="#F4F8FC"),
        gridcolor="#061A33"
    ),
    bargap=0.08,
    margin=dict(l=40, r=40, t=60, b=40)
)

fig.show()

In [15]:
fig = px.box(
    data,
    y="average score",
    points="outliers",
    title="Distribution of Students' Average Scores",
    labels={"average score": "Average Score"},
    color_discrete_sequence=["#008CFF"]
)

fig.update_traces(
    marker=dict(
        color="#00D9FF",
        size=6,
        line=dict(width=1, color="#008CFF")
    ),
    line=dict(color="#008CFF", width=2),
    fillcolor="#061A33"
)

fig.update_layout(
    plot_bgcolor="#020B18",
    paper_bgcolor="#020B18",
    font=dict(color="#F4F8FC", family="Arial, sans-serif"),
    title=dict(
        text="Distribution of Students' Average Scores",
        font=dict(color="#00D9FF", size=20)
    ),
    yaxis=dict(
        title=dict(text="Average Score", font=dict(color="#00D9FF")),
        tickfont=dict(color="#F4F8FC"),
        gridcolor="#061A33",
        zeroline=False
    ),
    xaxis=dict(
        tickfont=dict(color="#F4F8FC"),
        gridcolor="#061A33",
        showticklabels=False
    ),
    margin=dict(l=40, r=40, t=60, b=40)
)

fig.show()

<div style="background: linear-gradient(90deg, #061A33 0%, #020B18 100%); padding: 14px 20px; border-radius: 8px; text-align: left; border-left: 4px solid #008CFF;">

  <h3 style="color: #00D9FF; margin: 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial; font-weight: 700;">
    4.3 PARENTAL LEVEL OF EDUCATION
  </h3>

</div>



In [16]:
education_avg = (
    data.groupby("parental level of education")["total score"]
        .mean()
        .sort_values(ascending=False)
        .reset_index()
)

education_avg

,parental level of education,total score
0,master's degree,220.80
1,bachelor's degree,215.77
2,associate's degree,208.71
3,some college,205.43
4,some high school,195.32
5,high school,189.29


In [17]:
fig = px.bar(
    education_avg,
    x="parental level of education",
    y="total score",
    title="Average Total Score by Parental Education",
    labels={
        "parental level of education": "Parental Education",
        "total score": "Average Total Score"
    },
    text="total score",
    color_discrete_sequence=["#008CFF"]
)

fig.update_traces(
    texttemplate="%{text:.2f}",
    textposition="outside",
    marker=dict(
        line=dict(width=1, color="#00D9FF")
    )
)

fig.update_layout(
    plot_bgcolor="#020B18",
    paper_bgcolor="#020B18",
    font=dict(color="#F4F8FC", family="Arial, sans-serif"),
    title=dict(
        text="Average Total Score by Parental Education",
        font=dict(color="#00D9FF", size=20)
    ),
    xaxis=dict(
        title=dict(text="Parental Education", font=dict(color="#00D9FF")),
        tickfont=dict(color="#F4F8FC"),
        gridcolor="#061A33",
        tickangle=-25
    ),
    yaxis=dict(
        title=dict(text="Average Total Score", font=dict(color="#00D9FF")),
        tickfont=dict(color="#F4F8FC"),
        gridcolor="#061A33"
    ),
    margin=dict(l=40, r=40, t=60, b=80)
)

fig.show()

<div style="background: linear-gradient(135deg, #020B18 0%, #061A33 100%); padding: 20px; border-radius: 12px; border: 1px solid #008CFF;">

  <h3 style="color: #00D9FF; margin-top: 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial;">
    OBSERVATION
  </h3>

  <p style="color: #F4F8FC; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial; line-height: 1.6;">
    <strong>Students whose parents hold a master's degree have the highest average total score in this dataset, while the high-school group has the lowest average.</strong>
  </p>

  <p style="color: #F4F8FC; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial;">
    <strong style="color: #00D9FF;">Highest Average:</strong> Master's Degree
  </p>

  <p style="color: #F4F8FC; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial;">
    <strong style="color: #00D9FF;">Lowest Average:</strong> High School
  </p>

</div>

<div style="background: linear-gradient(90deg, #061A33 0%, #020B18 100%); padding: 14px 20px; border-radius: 8px; text-align: left; border-left: 4px solid #008CFF;">

  <h3 style="color: #00D9FF; margin: 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial; font-weight: 700;">
    4.4 PERFORMANCE BY GENDER
  </h3>

</div>



In [18]:
gender_scores = (
    data.groupby("gender")[score_columns]
        .mean()
        .reset_index()
)

gender_scores["average score"] = gender_scores[score_columns].mean(axis=1)

gender_scores

,gender,math score,reading score,writing score,average score
0,female,63.63,72.61,72.47,69.57
1,male,68.73,65.47,63.31,65.84


In [19]:
fig = px.bar(
    gender_scores,
    x="gender",
    y=score_columns,
    barmode="group",
    title="Average Subject Scores by Gender",
    labels={
        "gender": "Gender",
        "value": "Average Score",
        "variable": "Subject"
    },
    text_auto=".2f",
    color_discrete_sequence=["#008CFF", "#00D9FF", "#4A90D9"]
)

fig.update_traces(
    textposition="outside",
    marker=dict(
        line=dict(width=1, color="#F4F8FC")
    )
)

fig.update_layout(
    plot_bgcolor="#020B18",
    paper_bgcolor="#020B18",
    font=dict(color="#F4F8FC", family="Arial, sans-serif"),
    title=dict(
        text="Average Subject Scores by Gender",
        font=dict(color="#00D9FF", size=20)
    ),
    xaxis=dict(
        title=dict(text="Gender", font=dict(color="#00D9FF")),
        tickfont=dict(color="#F4F8FC"),
        gridcolor="#061A33"
    ),
    yaxis=dict(
        title=dict(text="Average Score", font=dict(color="#00D9FF")),
        tickfont=dict(color="#F4F8FC"),
        gridcolor="#061A33"
    ),
    legend=dict(
        title=dict(text="Subject", font=dict(color="#00D9FF")),
        font=dict(color="#F4F8FC"),
        bgcolor="#020B18",
        bordercolor="#008CFF",
        borderwidth=1
    ),
    margin=dict(l=40, r=40, t=60, b=40)
)

fig.show()

In [20]:
gender_overall = (
    data.groupby("gender")["average score"]
        .mean()
        .reset_index()
)

fig = px.bar(
    gender_overall,
    x="gender",
    y="average score",
    title="Overall Average Score by Gender",
    labels={"average score": "Average Score"},
    text="average score",
    color_discrete_sequence=["#008CFF"]
)

fig.update_traces(
    texttemplate="%{text:.2f}",
    textposition="outside",
    marker=dict(
        line=dict(width=1, color="#00D9FF")
    )
)

fig.update_layout(
    plot_bgcolor="#020B18",
    paper_bgcolor="#020B18",
    font=dict(color="#F4F8FC", family="Arial, sans-serif"),
    title=dict(
        text="Overall Average Score by Gender",
        font=dict(color="#00D9FF", size=20)
    ),
    xaxis=dict(
        title=dict(text="Gender", font=dict(color="#00D9FF")),
        tickfont=dict(color="#F4F8FC"),
        gridcolor="#061A33"
    ),
    yaxis=dict(
        title=dict(text="Average Score", font=dict(color="#00D9FF")),
        tickfont=dict(color="#F4F8FC"),
        gridcolor="#061A33"
    ),
    margin=dict(l=40, r=40, t=60, b=40)
)

fig.show()

<div style="background: linear-gradient(135deg, #020B18 0%, #061A33 100%); padding: 20px; border-radius: 12px; border: 1px solid #008CFF;">

  <h3 style="color: #00D9FF; margin-top: 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial;">
    OBSERVATION
  </h3>

  <p style="color: #F4F8FC; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial; line-height: 1.6;">
    <strong>Female students have higher average Reading and Writing scores, while male students have a higher average Math score. Overall, the female group has the higher average total score.</strong>
  </p>

  <p style="color: #F4F8FC; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial;">
    <strong style="color: #00D9FF;">Female Stronger In:</strong> Reading & Writing
  </p>

  <p style="color: #F4F8FC; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial;">
    <strong style="color: #00D9FF;">Male Stronger In:</strong> Math
  </p>

  <p style="color: #F4F8FC; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial;">
    <strong style="color: #00D9FF;">Higher Overall Average:</strong> Female students
  </p>

</div>

<div style="background: linear-gradient(90deg, #061A33 0%, #020B18 100%); padding: 14px 20px; border-radius: 8px; text-align: left; border-left: 4px solid #008CFF;">

  <h3 style="color: #00D9FF; margin: 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial; font-weight: 700;">
    4.5 TEST PREPARATION VS. TOTAL SCORE
  </h3>

</div>



In [21]:
prep_scores = (
    data.groupby("test preparation course")["total score"]
        .mean()
        .reset_index()
)

prep_scores

,test preparation course,total score
0,completed,218.01
1,none,195.12


In [22]:
fig = px.bar(
    prep_scores,
    x="test preparation course",
    y="total score",
    title="Average Total Score by Test-Preparation Status",
    labels={
        "test preparation course": "Course Status",
        "total score": "Average Total Score"
    },
    text="total score",
    color_discrete_sequence=["#008CFF"]
)

fig.update_traces(
    texttemplate="%{text:.2f}",
    textposition="outside",
    marker=dict(
        line=dict(width=1, color="#00D9FF")
    )
)

fig.update_layout(
    plot_bgcolor="#020B18",
    paper_bgcolor="#020B18",
    font=dict(color="#F4F8FC", family="Arial, sans-serif"),
    title=dict(
        text="Average Total Score by Test-Preparation Status",
        font=dict(color="#00D9FF", size=20)
    ),
    xaxis=dict(
        title=dict(text="Course Status", font=dict(color="#00D9FF")),
        tickfont=dict(color="#F4F8FC"),
        gridcolor="#061A33"
    ),
    yaxis=dict(
        title=dict(text="Average Total Score", font=dict(color="#00D9FF")),
        tickfont=dict(color="#F4F8FC"),
        gridcolor="#061A33"
    ),
    margin=dict(l=40, r=40, t=60, b=40)
)

fig.show()

<div style="background: linear-gradient(135deg, #020B18 0%, #061A33 100%); padding: 20px; border-radius: 12px; border: 1px solid #008CFF;">

  <h3 style="color: #00D9FF; margin-top: 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial;">
    OBSERVATION
  </h3>

  <p style="color: #F4F8FC; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial; line-height: 1.6;">
    <strong>Students who completed the test-preparation course have a higher average total score than students who did not.</strong>
  </p>

  <p style="color: #F4F8FC; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial;">
    <strong style="color: #00D9FF;">Completed Course:</strong> Higher average total score
  </p>

  <p style="color: #F4F8FC; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial;">
    <strong style="color: #00D9FF;">Did Not Complete:</strong> Lower average total score
  </p>

</div>

<div style="background: linear-gradient(90deg, #061A33 0%, #020B18 100%); padding: 14px 20px; border-radius: 8px; text-align: left; border-left: 4px solid #008CFF;">

  <h3 style="color: #00D9FF; margin: 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial; font-weight: 700;">
    4.6 RELATIONSHIP BETWEEN MATH AND READING SCORES
  </h3>

</div>



In [23]:
fig = px.scatter(
    data,
    x="math score",
    y="reading score",
    trendline="ols",
    title="Math Score vs. Reading Score",
    labels={
        "math score": "Math Score",
        "reading score": "Reading Score"
    },
    opacity=0.65,
    color_discrete_sequence=["#008CFF"]
)

fig.update_traces(
    marker=dict(
        size=8,
        line=dict(width=0.5, color="#00D9FF")
    ),
    selector=dict(mode='markers')
)

fig.update_traces(
    line=dict(color="#00D9FF", width=2),
    selector=dict(mode='lines')
)

fig.update_layout(
    plot_bgcolor="#020B18",
    paper_bgcolor="#020B18",
    font=dict(color="#F4F8FC", family="Arial, sans-serif"),
    title=dict(
        text="Math Score vs. Reading Score",
        font=dict(color="#00D9FF", size=20)
    ),
    xaxis=dict(
        title=dict(text="Math Score", font=dict(color="#00D9FF")),
        tickfont=dict(color="#F4F8FC"),
        gridcolor="#061A33",
        range=[0, 100]
    ),
    yaxis=dict(
        title=dict(text="Reading Score", font=dict(color="#00D9FF")),
        tickfont=dict(color="#F4F8FC"),
        gridcolor="#061A33",
        range=[0, 100]
    ),
    margin=dict(l=40, r=40, t=60, b=40)
)

fig.show()

<div style="background: linear-gradient(90deg, #061A33 0%, #020B18 100%); padding: 14px 20px; border-radius: 8px; text-align: left; border-left: 4px solid #008CFF;">

  <h3 style="color: #00D9FF; margin: 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial; font-weight: 700;">
    4.7 PERFORMANCE BY RACE/ETHNICITY GROUP
  </h3>

</div>



In [24]:
race_scores = (
    data.groupby("race/ethnicity")[score_columns]
        .mean()
        .reset_index()
)

race_scores

,race/ethnicity,math score,reading score,writing score
0,group A,61.63,64.67,62.67
1,group B,63.45,67.35,65.60
2,group C,64.46,69.10,67.83
3,group D,67.36,70.03,70.15
4,group E,73.82,73.03,71.41


In [25]:
fig = px.bar(
    race_scores,
    x="race/ethnicity",
    y=score_columns,
    barmode="group",
    title="Average Subject Scores by Race/Ethnicity",
    labels={
        "race/ethnicity": "Race/Ethnicity Group",
        "value": "Average Score",
        "variable": "Subject"
    },
    text_auto=".2f",
    color_discrete_sequence=["#008CFF", "#00D9FF", "#4A90D9"]
)

fig.update_traces(
    textposition="outside",
    marker=dict(
        line=dict(width=1, color="#F4F8FC")
    )
)

fig.update_layout(
    plot_bgcolor="#020B18",
    paper_bgcolor="#020B18",
    font=dict(color="#F4F8FC", family="Arial, sans-serif"),
    title=dict(
        text="Average Subject Scores by Race/Ethnicity",
        font=dict(color="#00D9FF", size=20)
    ),
    xaxis=dict(
        title=dict(text="Race/Ethnicity Group", font=dict(color="#00D9FF")),
        tickfont=dict(color="#F4F8FC"),
        gridcolor="#061A33"
    ),
    yaxis=dict(
        title=dict(text="Average Score", font=dict(color="#00D9FF")),
        tickfont=dict(color="#F4F8FC"),
        gridcolor="#061A33"
    ),
    legend=dict(
        title=dict(text="Subject", font=dict(color="#00D9FF")),
        font=dict(color="#F4F8FC"),
        bgcolor="#020B18",
        bordercolor="#008CFF",
        borderwidth=1
    ),
    margin=dict(l=40, r=40, t=60, b=40)
)

fig.show()

<div style="background: linear-gradient(135deg, #020B18 0%, #061A33 100%); padding: 20px; border-radius: 12px; border: 1px solid #008CFF;">

  <h3 style="color: #00D9FF; margin-top: 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial;">
    OBSERVATION
  </h3>

  <p style="color: #F4F8FC; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial; line-height: 1.6;">
    <strong> Group E has the highest average performance across Math, Reading, and Writing, while Group A has the lowest averages among the five groups in this dataset.
  </p>

  <p style="color: #F4F8FC; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial;">
    <strong style="color: #00D9FF;">Highest Performing:</strong> Group E
  </p>

  <p style="color: #F4F8FC; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial;">
    <strong style="color: #00D9FF;">Lowest Performing:</strong> Group A
  </p>

</div>

<div style="background: linear-gradient(90deg, #020B18 0%, #061A33 55%, #008CFF 100%); padding: 18px 22px; border-radius: 10px; text-align: left; border-left: 5px solid #00D9FF;">

  <h2 style="color: #F4F8FC; margin: 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial; font-weight: 700;">
    🔎 5. OUTLIER ANALYSIS
  </h2>

</div>



In [26]:
Q1 = data["total score"].quantile(0.25)
Q3 = data["total score"].quantile(0.75)
IQR = Q3 - Q1

lower_bound = Q1 - 1.5 * IQR
upper_bound = Q3 + 1.5 * IQR

outliers = data[
    (data["total score"] < lower_bound) |
    (data["total score"] > upper_bound)
]

print(f"Q1: {Q1:.2f}")
print(f"Q3: {Q3:.2f}")
print(f"IQR: {IQR:.2f}")
print(f"Lower bound: {lower_bound:.2f}")
print(f"Upper bound: {upper_bound:.2f}")
print(f"Number of outliers: {len(outliers)}")

Q1: 175.00
Q3: 233.00
IQR: 58.00
Lower bound: 88.00
Upper bound: 320.00
Number of outliers: 6


In [27]:
outliers[
    [
        "gender",
        "race/ethnicity",
        "lunch",
        "test preparation course",
        *score_columns,
        "average score",
        "total score"
    ]
].sort_values("total score")

,gender,race/ethnicity,lunch,test preparation course,math score,reading score,writing score,average score,total score
59,female,group C,free/reduced,none,0,17,10,9.00,27
980,female,group B,free/reduced,none,8,24,23,18.33,55
596,male,group B,free/reduced,none,30,24,15,23.00,69
327,male,group A,free/reduced,none,28,23,19,23.33,70
76,male,group E,standard,none,30,26,22,26.00,78
17,female,group B,free/reduced,none,18,32,28,26.00,78


<div style="background: linear-gradient(135deg, #020B18 0%, #061A33 100%); padding: 20px; border-radius: 12px; border: 1px solid #008CFF;">

  <h3 style="color: #00D9FF; margin-top: 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial;">
    INTERPRETATION
  </h3>

  <p style="color: #F4F8FC; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial; line-height: 1.6;">
    The IQR method identifies unusually low total scores. These observations are not automatically errors; they may represent genuinely low-performing students and should therefore be investigated rather than removed without justification.
  </p>

</div>

<div style="background: linear-gradient(90deg, #020B18 0%, #061A33 55%, #008CFF 100%); padding: 18px 22px; border-radius: 10px; text-align: left; border-left: 5px solid #00D9FF;">

  <h2 style="color: #F4F8FC; margin: 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial; font-weight: 700;">
    🧪 6. HYPOTHESIS TESTING — DOES TEST PREPARATION MATTER?
  </h2>

</div>



<div style="background: linear-gradient(90deg, #020B18 0%, #061A33 100%); padding: 20px; border-radius: 12px; border-left: 5px solid #008CFF;">

  <h3 style="color: #00D9FF; margin-top: 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial;">
    RESEARCH QUESTION 
  </h3>

  <p style="color: #F4F8FC; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial; line-height: 1.8;">
    Do students who completed the test-preparation course have a different mean total score from students who did not?
  </p>

  <ul style="color: #F4F8FC; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial; line-height: 1.8;">
    <li><strong style="color: #00D9FF;">Null hypothesis (H₀):</strong> The mean total score is the same for both groups.</li>
    <li><strong style="color: #00D9FF;">Alternative hypothesis (H₁):</strong> The mean total score differs between the two groups.</li>
  </ul>

  <p style="color: #F4F8FC; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial; line-height: 1.8;">
    We use an <strong style="color: #00D9FF;">independent two-sample Welch's t-test</strong>, which does not assume equal population variances.
  </p>

</div>

In [28]:
completed = data.loc[
    data["test preparation course"] == "completed",
    "total score"
]

not_completed = data.loc[
    data["test preparation course"] == "none",
    "total score"
]

t_stat, p_value = ttest_ind(
    completed,
    not_completed,
    equal_var=False
)

print(f"Completed group mean: {completed.mean():.2f}")
print(f"Not-completed group mean: {not_completed.mean():.2f}")
print(f"t-statistic: {t_stat:.4f}")
print(f"p-value: {p_value:.4e}")

Completed group mean: 218.01
Not-completed group mean: 195.12
t-statistic: 8.5945
p-value: 4.4267e-17


In [29]:
alpha = 0.05

if p_value < alpha:
    print(
        "Decision: Reject H₀. "
        "There is statistically significant evidence of a difference in mean total scores."
    )
else:
    print(
        "Decision: Fail to reject H₀. "
        "There is not enough evidence of a difference in mean total scores."
    )

Decision: Reject H₀. There is statistically significant evidence of a difference in mean total scores.


<div style="background: linear-gradient(135deg, #020B18 0%, #061A33 100%); padding: 20px; border-radius: 12px; border: 1px solid #008CFF;">

  <h3 style="color: #00D9FF; margin-top: 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial;">
    STATISTICAL CONCLUSION
  </h3>

  <p style="color: #F4F8FC; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial; line-height: 1.6;">
    The p-value is far below the 0.05 significance level, so we <strong style="color: #00D9FF;">reject the null hypothesis</strong>. There is strong statistical evidence that the mean total scores differ between students who completed the preparation course and those who did not.
  </p>

  <p style="color: #F4F8FC; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial; line-height: 1.6;">
    <strong style="color: #00D9FF;">Important:</strong> Statistical significance shows that the groups differ; it does <strong>not</strong> by itself prove that completing the course caused the higher scores. Other factors may also contribute to the difference.
  </p>

</div>

<div style="background: linear-gradient(90deg, #020B18 0%, #061A33 55%, #008CFF 100%); padding: 18px 22px; border-radius: 10px; text-align: left; border-left: 5px solid #00D9FF;">

  <h2 style="color: #F4F8FC; margin: 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial; font-weight: 700;">
    📌 7. KEY FINDINGS
  </h2>

</div>


<div style="background: linear-gradient(135deg, #020B18 0%, #061A33 100%); padding: 20px; border-radius: 12px; border: 1px solid #008CFF;">

  <h3 style="color: #00D9FF; margin-top: 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial;">
    MAIN INSIGHTS
  </h3>

  <ul style="color: #F4F8FC; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial; line-height: 2.0; padding-left: 20px;">
    <li><strong style="color: #00D9FF;">Data quality:</strong> No missing values were found in the original dataset.</li>
    <li><strong style="color: #00D9FF;">Overall performance:</strong> Reading and Writing have higher average scores than Math.</li>
    <li><strong style="color: #00D9FF;">Test preparation:</strong> 358 students completed the course, compared with 642 who did not.</li>
    <li><strong style="color: #00D9FF;">Course performance:</strong> Students who completed the preparation course have a higher average total score.</li>
    <li><strong style="color: #00D9FF;">Hypothesis test:</strong> Welch's t-test indicates a statistically significant difference in total scores between the two groups.</li>
    <li><strong style="color: #00D9FF;">Gender:</strong> Female students perform better on average in Reading and Writing, while male students perform better in Math.</li>
    <li><strong style="color: #00D9FF;">Parental education:</strong> The master's-degree group has the highest average total score.</li>
    <li><strong style="color: #00D9FF;">Race/ethnicity:</strong> Group E has the highest average across the three subjects, while Group A has the lowest.</li>
    <li><strong style="color: #00D9FF;">Outliers:</strong> The IQR method identifies a small number of unusually low total scores; these should be investigated rather than automatically removed.</li>
  </ul>

  <p style="color: #F4F8FC; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial; line-height: 1.8; margin-top: 16px; border-top: 1px solid #008CFF; padding-top: 16px;">
    <strong style="color: #00D9FF;">Final takeaway:</strong> The analysis suggests that <strong style="color: #00D9FF;">test preparation, student demographics, and parental education are associated with differences in academic performance</strong> in this dataset. The strongest statistical result explored here is the significant difference in total scores between students who completed the test-preparation course and those who did not.
  </p>

</div>

<div style="background: linear-gradient(90deg, #020B18 0%, #061A33 60%, #008CFF 100%); padding: 18px 22px; border-radius: 10px; border-left: 5px solid #00D9FF;">

  <h3 style="color: #00D9FF; margin: 0 0 10px 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial;">
    NOTES & LIMITATIONS
  </h3>

  <ul style="color: #F4F8FC; margin: 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial; line-height: 1.8; padding-left: 20px;">
    <li>This is an <strong style="color: #00D9FF;">observational dataset</strong>, so statistical association should not be interpreted as causation.</li>
    <li>The race/ethnicity and gender categories are treated as the labels provided by the dataset; the analysis does not make broader claims about demographic groups.</li>
    <li>Outliers are retained because extreme scores can represent real observations.</li>
    <li>The hypothesis test compares group means and does not control for other variables such as lunch type, parental education, or gender.</li>
  </ul>

</div>

<div style="background: linear-gradient(135deg, #020B18 0%, #061A33 100%); padding: 20px; border-radius: 12px; border: 1px solid #008CFF;">

  <h3 style="color: #00D9FF; margin-top: 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial;">
    🛠️ Tools Used
  </h3>

  <p style="color: #F4F8FC; margin-bottom: 0; font-family: -apple-system, BlinkMacSystemFont, 'Segoe UI', Roboto, Arial; line-height: 1.6;">
    Python · Pandas · NumPy · Matplotlib · Seaborn · Plotly · SciPy
  </p>

</div>